In [1]:
# ================================================================
# NOTEBOOK 14: CWQ ANSWER-NODE RECOGNIZER
# Runtime: T4 GPU
#
# - Uses existing CWQ files.
# - Trains a separate CWQ MiniLM model.
# - Preserves the trained CWQ controller and its threshold.
# - Uses development data only for answer-threshold selection.
# - Does not perform test-set evaluation.
# ================================================================

import collections
import hashlib
import importlib
import importlib.metadata
import inspect
import json
import math
import os
import random
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import numpy as np


# ---------------------------- SETTINGS ----------------------------

ROOT = Path("/content/drive/MyDrive/SetStop_KGQA")
MODEL_NAME = "microsoft/MiniLM-L12-H384-uncased"

SEED = 42
MAX_HOPS = 4
MAX_LENGTH = 192

NEGATIVES_PER_POSITIVE = 3
CHECKPOINT_DEV_SIZE = 8000

TRAIN_BATCH = 32
EVAL_BATCH = 128
EPOCHS = 2

LEARNING_RATE = 2e-5
EVAL_STEPS = 500
PREDICTION_CHUNK_QUESTIONS = 25

CODE_VERSION = "cwq_answer_minilm_v1_0"


# ------------------------- FILE UTILITIES -------------------------

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def save_json(path, obj):
    temporary = path.with_name(path.name + ".partial")
    with open(temporary, "w", encoding="utf-8") as file:
        json.dump(
            obj,
            file,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )
    temporary.replace(path)


def rows(path):
    with open(path, encoding="utf-8") as file:
        for line in file:
            if line.strip():
                yield json.loads(line)


def file_hash(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for block in iter(
            lambda: file.read(8 * 1024**2),
            b"",
        ):
            digest.update(block)
    return digest.hexdigest()


def fingerprint(obj):
    return hashlib.sha256(
        json.dumps(obj, sort_keys=True).encode()
    ).hexdigest()


def verify_inventory(path, inventory, name=None):
    name = name or path.name

    entry = next(
        (
            item
            for item in inventory["files"]
            if item["name"] == name
        ),
        None,
    )

    if (
        entry is None
        or path.stat().st_size != entry["bytes"]
        or file_hash(path) != entry["sha256"]
    ):
        raise ValueError(
            f"Inventory verification failed: {path}"
        )


# ----------------------- ANSWER-NODE INPUT ------------------------
# Matches the WebQSP answer-recognizer text format.
# Gold labels and entity IDs are not included in model text.

def answer_text(
    question,
    path,
    target_count,
    outgoing_count,
    relation_names,
):
    path_text = " -> ".join(
        relation_names[relation]
        for relation in path
    )

    return (
        f"Question: {question}\n"
        f"Candidate relation path: {path_text}\n"
        f"Candidate depth: {len(path)}\n"
        f"Incoming relation: {relation_names[path[-1]]}\n"
        f"Entities returned by final relation: {target_count}\n"
        f"Candidate outgoing relation count: {outgoing_count}"
    )


def graph_candidates(
    row,
    entity_to_id,
    relation_to_id,
    relation_names,
):
    def entity(value):
        if isinstance(value, dict):
            value = value.get(
                "kb_id",
                value.get("id", value.get("entity")),
            )

        if isinstance(value, int) and not isinstance(value, bool):
            result = (
                value
                if 0 <= value < len(entity_to_id)
                else None
            )
        else:
            result = (
                entity_to_id.get(value)
                if isinstance(value, str)
                else None
            )

        if result is None:
            raise ValueError(f"Unmapped entity: {value}")

        return result

    def relation(value):
        if isinstance(value, int) and not isinstance(value, bool):
            result = (
                value
                if 0 <= value < len(relation_names)
                else None
            )
        else:
            result = (
                relation_to_id.get(value)
                if isinstance(value, str)
                else None
            )

        if result is None:
            raise ValueError(f"Unmapped relation: {value}")

        return result

    qid = str(row["id"])
    question = row["question"].strip()
    subgraph = row["subgraph"]

    nodes = {
        entity(value)
        for value in subgraph["entities"]
    }

    triples = subgraph.get(
        "tuples",
        subgraph.get("triples"),
    )

    if not isinstance(triples, list):
        raise ValueError(f"Missing graph triples: {qid}")

    adjacency = collections.defaultdict(
        lambda: collections.defaultdict(set)
    )

    for edge in triples:
        if (
            not isinstance(edge, (list, tuple))
            or len(edge) != 3
        ):
            raise ValueError(f"Invalid graph edge: {qid}")

        source = entity(edge[0])
        rel = relation(edge[1])
        target = entity(edge[2])

        adjacency[source][rel].add(target)
        nodes.update([source, target])

    topics = {
        entity(value)
        for value in row.get(
            "entities",
            row.get("topic_entities", []),
        )
    }

    gold = {
        entity(value)
        for value in row["answers"]
    }

    if not (
        topics
        and topics <= nodes
        and gold <= nodes
        and 2 <= len(gold) <= 50
    ):
        raise ValueError(
            f"Question no longer satisfies CWQ eligibility: {qid}"
        )

    # Deterministic shortest paths, selected without using gold.
    paths = {
        node: []
        for node in sorted(topics)
    }

    queue = collections.deque(sorted(topics))
    candidates = []

    while queue:
        source = queue.popleft()
        source_path = paths[source]

        if len(source_path) >= MAX_HOPS:
            continue

        for rel, targets in sorted(
            adjacency.get(source, {}).items()
        ):
            for target in sorted(targets):
                if target in paths:
                    continue

                path = source_path + [rel]
                paths[target] = path
                queue.append(target)

                candidates.append({
                    "entity": target,
                    "depth": len(path),
                    "text": answer_text(
                        question,
                        path,
                        len(targets),
                        len(adjacency.get(target, {})),
                        relation_names,
                    ),
                    "label": int(target in gold),
                })

    if not gold <= paths.keys():
        raise ValueError(
            f"A selected gold answer is not reachable "
            f"within {MAX_HOPS} hops: {qid}"
        )

    return {
        "qid": qid,
        "gold_ids": sorted(gold),
        "candidates": candidates,
        "topic_gold_count": len(gold & topics),
        "reachable_nodes": len(paths),
    }


# ------------------------ DATASET BUILDING ------------------------

def build_split(
    split,
    selected,
    source,
    expected_hash,
    cache,
    signature,
    entity_to_id,
    relation_to_id,
    relation_names,
):
    path = cache / f"{split}_answer_candidates_v1.jsonl"
    metadata_path = cache / f"{split}_answer_candidates_v1.json"

    specification = {
        "signature": signature,
        "split": split,
        "source_sha256": expected_hash,
        "qids": [item["qid"] for item in selected],
        "code_version": CODE_VERSION,
        "max_hops": MAX_HOPS,
        "negatives_per_positive": NEGATIVES_PER_POSITIVE,
    }

    cache_signature = fingerprint(specification)

    metadata = (
        read_json(metadata_path)
        if metadata_path.exists() and path.exists()
        else None
    )

    if (
        metadata
        and metadata["signature"] == cache_signature
        and metadata["sha256"] == file_hash(path)
    ):
        print(
            f"{split}: reused candidate cache.",
            flush=True,
        )
        return path, metadata

    by_line = {
        int(item["line_number"]): item
        for item in selected
    }

    if len(by_line) != len(selected):
        raise ValueError("Duplicate selected line number.")

    statistics = collections.Counter(
        questions=0,
        gold_answers=0,
        candidate_gold_answers=0,
        topic_gold_answers=0,
        reachable_nodes=0,
        available_candidates=0,
        positive_examples=0,
        negative_examples=0,
    )

    completed = set()
    temporary = path.with_name(path.name + ".partial")
    started = time.time()

    with (
        open(source, encoding="utf-8") as source_file,
        open(temporary, "w", encoding="utf-8") as target_file,
    ):
        for line_number, line in enumerate(source_file, 1):
            if line_number not in by_line:
                continue

            entry = by_line[line_number]
            raw = json.loads(line)

            qid = str(raw["id"])
            question_hash = hashlib.sha256(
                " ".join(
                    raw["question"].lower().split()
                ).encode()
            ).hexdigest()

            if (
                qid != entry["qid"]
                or question_hash != entry["question_sha256"]
                or qid in completed
            ):
                raise ValueError(
                    f"Manifest/source disagreement: {qid}"
                )

            group = graph_candidates(
                raw,
                entity_to_id,
                relation_to_id,
                relation_names,
            )

            if len(group["gold_ids"]) != entry["gold_count"]:
                raise ValueError(f"Gold count changed: {qid}")

            available = group["candidates"]

            positives = [
                candidate
                for candidate in available
                if candidate["label"] == 1
            ]

            negatives = [
                candidate
                for candidate in available
                if candidate["label"] == 0
            ]

            if split == "train":
                question_seed = int(
                    hashlib.sha256(
                        qid.encode()
                    ).hexdigest()[:8],
                    16,
                )

                rng = random.Random(SEED + question_seed)

                negative_count = min(
                    len(negatives),
                    NEGATIVES_PER_POSITIVE
                    * max(1, len(positives)),
                )

                candidates = (
                    positives
                    + rng.sample(negatives, negative_count)
                )

                rng.shuffle(candidates)
                group["candidates"] = candidates

            else:
                # Retain every development candidate.
                candidates = available

            statistics["questions"] += 1
            statistics["gold_answers"] += len(group["gold_ids"])
            statistics["candidate_gold_answers"] += len(positives)
            statistics["topic_gold_answers"] += group["topic_gold_count"]
            statistics["reachable_nodes"] += group["reachable_nodes"]
            statistics["available_candidates"] += len(available)

            statistics["positive_examples"] += sum(
                candidate["label"]
                for candidate in candidates
            )

            statistics["negative_examples"] += sum(
                1 - candidate["label"]
                for candidate in candidates
            )

            completed.add(qid)

            target_file.write(
                json.dumps(
                    group,
                    ensure_ascii=False,
                ) + "\n"
            )

            if len(completed) % 200 == 0:
                print(
                    f"{split}: {len(completed):,}/"
                    f"{len(selected):,} questions | "
                    f"{(time.time() - started) / 60:.1f} min",
                    flush=True,
                )

    expected_qids = {
        item["qid"]
        for item in selected
    }

    if completed != expected_qids:
        raise ValueError(
            f"Missing selected {split} questions."
        )

    temporary.replace(path)

    metadata = {
        "signature": cache_signature,
        "specification": specification,
        "statistics": dict(statistics),
        "examples": (
            statistics["positive_examples"]
            + statistics["negative_examples"]
        ),
        "bytes": path.stat().st_size,
        "sha256": file_hash(path),
    }

    save_json(metadata_path, metadata)

    return path, metadata


def checkpoint_samples(train_path, dev_path):
    train = [
        {
            "text": candidate["text"],
            "label": candidate["label"],
        }
        for group in rows(train_path)
        for candidate in group["candidates"]
    ]

    random.Random(SEED).shuffle(train)

    dev = []
    seen = 0
    rng = random.Random(SEED + 100)

    for group in rows(dev_path):
        for candidate in group["candidates"]:
            seen += 1

            item = {
                "text": candidate["text"],
                "label": candidate["label"],
            }

            if len(dev) < CHECKPOINT_DEV_SIZE:
                dev.append(item)
            else:
                index = rng.randrange(seen)
                if index < CHECKPOINT_DEV_SIZE:
                    dev[index] = item

    random.Random(SEED).shuffle(dev)

    if (
        not train
        or not dev
        or {item["label"] for item in train} != {0, 1}
    ):
        raise ValueError(
            "Examples are empty or a training class is missing."
        )

    return train, dev


# ----------------------------- METRICS ----------------------------

def binary_metrics(labels, probabilities, threshold):
    truth = np.asarray(labels, dtype=bool)
    predicted = np.asarray(probabilities) >= threshold

    tp = int(np.sum(truth & predicted))
    fp = int(np.sum(~truth & predicted))
    fn = int(np.sum(truth & ~predicted))
    tn = int(np.sum(~truth & ~predicted))

    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)

    positive_f1 = 2 * tp / max(1, 2 * tp + fp + fn)
    negative_f1 = 2 * tn / max(1, 2 * tn + fp + fn)

    return {
        "accuracy": (tp + tn) / max(1, len(truth)),
        "balanced_accuracy": (
            recall + tn / max(1, tn + fp)
        ) / 2,
        "macro_f1": (positive_f1 + negative_f1) / 2,
        "answer_f1": positive_f1,
        "answer_precision": precision,
        "answer_recall": recall,
        "confusion_matrix": [[tn, fp], [fn, tp]],
        "positive_examples": int(truth.sum()),
        "negative_examples": int((~truth).sum()),
        "number_of_examples": len(truth),
        "classification_report": {
            "not_answer": {
                "precision": tn / max(1, tn + fn),
                "recall": tn / max(1, tn + fp),
                "f1": negative_f1,
                "support": tn + fp,
            },
            "answer": {
                "precision": precision,
                "recall": recall,
                "f1": positive_f1,
                "support": tp + fn,
            },
        },
    }


def question_threshold_sweep(groups):
    thresholds = np.linspace(0.0, 1.0, 101)

    keys = [
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "exact_match",
        "complete_recall",
        "tp",
        "fp",
        "fn",
    ]

    totals = {
        key: np.zeros(len(thresholds))
        for key in keys
    }

    for group in groups:
        values = np.asarray(
            group["probabilities"],
            dtype=float,
        )
        labels = np.asarray(
            group["labels"],
            dtype=bool,
        )

        total_gold = len(group["gold_ids"])

        predicted_count = (
            len(values)
            - np.searchsorted(
                np.sort(values),
                thresholds,
                side="left",
            )
        )

        tp = (
            int(labels.sum())
            - np.searchsorted(
                np.sort(values[labels]),
                thresholds,
                side="left",
            )
        )

        fp = predicted_count - tp
        fn = total_gold - tp

        totals["macro_precision"] += np.divide(
            tp,
            predicted_count,
            out=np.zeros(len(thresholds)),
            where=predicted_count > 0,
        )

        totals["macro_recall"] += tp / total_gold
        totals["macro_f1"] += (
            2 * tp / (predicted_count + total_gold)
        )

        totals["exact_match"] += (
            (tp == total_gold)
            & (predicted_count == total_gold)
        )

        totals["complete_recall"] += tp == total_gold
        totals["tp"] += tp
        totals["fp"] += fp
        totals["fn"] += fn

    sweep = []

    for index, threshold in enumerate(thresholds):
        result = {
            "threshold": float(threshold),
            "questions": len(groups),
        }

        for key in [
            "macro_precision",
            "macro_recall",
            "macro_f1",
            "exact_match",
            "complete_recall",
        ]:
            result[key] = float(
                totals[key][index] / max(1, len(groups))
            )

        tp, fp, fn = (
            int(totals[key][index])
            for key in ["tp", "fp", "fn"]
        )

        result.update(
            micro_f1=2 * tp / max(1, 2 * tp + fp + fn),
            true_positive_count=tp,
            false_positive_count=fp,
            false_negative_count=fn,
        )

        sweep.append(result)

    best = min(
        sweep,
        key=lambda result: (
            -result["macro_f1"],
            -result["complete_recall"],
            -result["macro_precision"],
            -result["threshold"],
        ),
    )

    return best, sweep


# ------------------- RESUMABLE DEVELOPMENT SCORING ----------------

def score_development(
    path,
    metadata,
    output,
    signature,
    predictor,
):
    chunks = (
        output
        / "dev_prediction_chunks"
        / signature[:12]
    )
    chunks.mkdir(parents=True, exist_ok=True)

    all_groups = []
    buffer = []
    chunk_index = 0
    started = time.time()

    def process(group_rows, index):
        chunk_path = chunks / f"chunk_{index:04d}.json"
        qids = [group["qid"] for group in group_rows]

        cached = (
            read_json(chunk_path)
            if chunk_path.exists()
            else None
        )

        if (
            cached
            and cached.get("signature") == signature
            and cached.get("qids") == qids
        ):
            return cached["groups"]

        texts = [
            candidate["text"]
            for group in group_rows
            for candidate in group["candidates"]
        ]

        scores = np.asarray(
            predictor(texts),
            dtype=np.float32,
        )

        if (
            scores.shape != (len(texts),)
            or not np.isfinite(scores).all()
            or np.any((scores < 0) | (scores > 1))
        ):
            raise ValueError(
                "Invalid answer probabilities."
            )

        result = []
        offset = 0

        for group in group_rows:
            candidates = group["candidates"]
            size = len(candidates)

            result.append({
                "qid": group["qid"],
                "gold_ids": group["gold_ids"],
                "entities": [
                    candidate["entity"]
                    for candidate in candidates
                ],
                "labels": [
                    candidate["label"]
                    for candidate in candidates
                ],
                "probabilities": (
                    scores[offset:offset + size].tolist()
                ),
            })

            offset += size

        save_json(
            chunk_path,
            {
                "signature": signature,
                "qids": qids,
                "groups": result,
            },
        )

        return result

    for group in rows(path):
        buffer.append(group)

        if len(buffer) == PREDICTION_CHUNK_QUESTIONS:
            all_groups.extend(
                process(buffer, chunk_index)
            )
            buffer = []
            chunk_index += 1

            print(
                f"Dev scoring: {len(all_groups):,}/"
                f'{metadata["statistics"]["questions"]:,} '
                f"questions | "
                f"{(time.time() - started) / 60:.1f} min",
                flush=True,
            )

    if buffer:
        all_groups.extend(
            process(buffer, chunk_index)
        )

    if (
        len(all_groups)
        != metadata["statistics"]["questions"]
        or sum(
            len(group["labels"])
            for group in all_groups
        ) != metadata["examples"]
    ):
        raise ValueError(
            "Development totals disagree with the candidate cache."
        )

    return all_groups


# ------------------------------ MAIN ------------------------------

def main():
    from google.colab import drive
    drive.mount("/content/drive")

    import torch

    if not torch.cuda.is_available():
        raise RuntimeError(
            "Select Runtime > Change runtime type > T4 GPU, "
            "then rerun."
        )

    os.environ["USE_TF"] = "0"
    os.environ["USE_FLAX"] = "0"

    # Preserve Colab's installed PyTorch.
    with tempfile.NamedTemporaryFile(
        mode="w",
        suffix=".txt",
    ) as constraint:
        constraint.write(
            "torch==" + torch.__version__.split("+")[0] + "\n"
        )
        constraint.flush()

        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "-c",
                constraint.name,
                "transformers>=4.57,<6",
                "accelerate>=1.12,<2",
                "safetensors>=0.6,<1",
                "tqdm>=4.60",
            ],
            check=True,
        )

    importlib.invalidate_caches()

    import transformers

    from transformers import (
        AutoConfig,
        AutoTokenizer,
        AutoModelForSequenceClassification,
        TrainingArguments,
        Trainer,
        TrainerCallback,
    )
    from tqdm.auto import tqdm

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    data = ROOT / "data/extracted/cwq"
    preprocessing = ROOT / "outputs/cwq_preprocessing_v1"
    controller_output = ROOT / "outputs/cwq_controller_training_v1"

    output = ROOT / "outputs/cwq_answer_recognizer_v1"
    cache = ROOT / "data/processed/cwq_answer_recognizer_cache_v1"
    run = ROOT / "models/cwq_answer_recognizer_training_v1"
    model_dir = ROOT / "models/cwq_answer_node_minilm_v1"

    for path in [output, cache, run]:
        path.mkdir(parents=True, exist_ok=True)

    generation = read_json(
        preprocessing / "cwq_state_generation_summary_v1.json"
    )
    state_inventory = read_json(
        preprocessing / "artifact_inventory_v1.json"
    )

    verify_inventory(
        preprocessing / "cwq_state_generation_summary_v1.json",
        state_inventory,
    )

    manifest_path = Path(generation["manifest_path"])
    verify_inventory(manifest_path, state_inventory)
    manifest = read_json(manifest_path)

    controller_inventory = read_json(
        controller_output / "artifact_inventory_v1.json"
    )

    for name in [
        "cwq_controller_training_summary_v1.json",
        "cwq_dev_stop_threshold_v1.json",
    ]:
        verify_inventory(
            controller_output / name,
            controller_inventory,
        )

    controller = read_json(
        controller_output / "cwq_controller_training_summary_v1.json"
    )
    controller_threshold = read_json(
        controller_output / "cwq_dev_stop_threshold_v1.json"
    )

    if not (
        generation["signature"]
        == manifest["signature"]
        == controller["data_signature"]
        == controller_inventory["data_signature"]
    ):
        raise ValueError(
            "CWQ preprocessing/controller signatures disagree."
        )

    if (
        generation["configuration"]["max_hops"] != MAX_HOPS
        or controller_threshold["test_used"]
    ):
        raise ValueError(
            "Invalid CWQ hop configuration or threshold provenance."
        )

    frozen_controller_threshold = float(
        controller_threshold["selected_threshold"]
    )

    selected = {
        split: manifest["splits"][split]
        for split in ["train", "dev"]
    }

    if (
        {item["qid"] for item in selected["train"]}
        & {item["qid"] for item in selected["dev"]}
    ):
        raise ValueError(
            "Train/development question overlap."
        )

    audit_path = (
        ROOT
        / "outputs/cwq_audit_v1/cwq_dataset_audit_summary_v1.json"
    )
    audit = read_json(audit_path)

    if (
        audit["signature"]
        != generation["configuration"]["audit_signature"]
    ):
        raise ValueError(
            "Source audit does not match Notebook 12."
        )

    source_hashes = audit["configuration"]["source_files"]

    print("=" * 90)
    print("NOTEBOOK 14: CWQ ANSWER-NODE RECOGNIZER")
    print("=" * 90)
    print("GPU:", torch.cuda.get_device_name(0))
    print("Transformers:", transformers.__version__)
    print("PyTorch:", torch.__version__)
    print("Model:", MODEL_NAME)
    print(
        "Frozen controller threshold:",
        frozen_controller_threshold,
    )
    print(
        "Selected train/dev questions:",
        len(selected["train"]),
        len(selected["dev"]),
    )

    # No test data are scanned or evaluated.
    for name in [
        "train_simple.json",
        "dev_simple.json",
        "entities.txt",
        "relations.txt",
    ]:
        print("Verifying source:", name, flush=True)

        path = data / name
        expected = source_hashes[name]

        if (
            path.stat().st_size != expected["bytes"]
            or file_hash(path) != expected["sha256"]
        ):
            raise ValueError(
                f"CWQ source changed: {name}"
            )

    with open(
        data / "entities.txt",
        encoding="utf-8",
    ) as file:
        entity_to_id = {
            line.rstrip("\r\n"): index
            for index, line in enumerate(file)
        }

    with open(
        data / "relations.txt",
        encoding="utf-8",
    ) as file:
        relation_names = [
            line.rstrip("\r\n")
            for line in file
        ]

    relation_to_id = {
        name: index
        for index, name in enumerate(relation_names)
    }

    paths = {}
    metadata = {}

    for split in ["train", "dev"]:
        paths[split], metadata[split] = build_split(
            split,
            selected[split],
            data / f"{split}_simple.json",
            source_hashes[f"{split}_simple.json"]["sha256"],
            cache,
            generation["signature"],
            entity_to_id,
            relation_to_id,
            relation_names,
        )

    del entity_to_id, relation_to_id

    print(
        "\nTraining statistics:",
        json.dumps(
            metadata["train"]["statistics"],
            indent=2,
        ),
    )
    print(
        "\nDevelopment statistics:",
        json.dumps(
            metadata["dev"]["statistics"],
            indent=2,
        ),
    )

    train_examples, dev_examples = checkpoint_samples(
        paths["train"],
        paths["dev"],
    )

    specification = {
        "code_version": CODE_VERSION,
        "model": MODEL_NAME,
        "data_signature": generation["signature"],
        "train_sha256": metadata["train"]["sha256"],
        "dev_sha256": metadata["dev"]["sha256"],
        "seed": SEED,
        "max_hops": MAX_HOPS,
        "max_length": MAX_LENGTH,
        "train_batch": TRAIN_BATCH,
        "eval_batch": EVAL_BATCH,
        "checkpoint_dev_size": CHECKPOINT_DEV_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "negatives_per_positive": NEGATIVES_PER_POSITIVE,
        "answer_text_version": "webqsp_notebook06_v1",
    }

    run_signature = fingerprint(specification)
    run_manifest_path = run / "run_manifest_v1.json"

    run_manifest = (
        read_json(run_manifest_path)
        if run_manifest_path.exists()
        else None
    )

    if (
        run_manifest
        and run_manifest["signature"] != run_signature
    ):
        raise ValueError(
            "Existing CWQ answer run uses different settings. "
            "Use a new run directory."
        )

    hf_cache = "/content/setstop_cwq_minilm_cache_v1"

    revision = (
        run_manifest["base_revision"]
        if run_manifest
        else None
    )

    base_config = AutoConfig.from_pretrained(
        MODEL_NAME,
        revision=revision or "main",
        token=False,
        cache_dir=hf_cache,
    )
    revision = revision or base_config._commit_hash

    if not revision:
        raise ValueError(
            "Could not resolve the base-model revision."
        )

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        revision=revision,
        token=False,
        cache_dir=hf_cache,
    )

    tokenizer.padding_side = "right"
    tokenizer.truncation_side = "right"

    if run_manifest is None:
        save_json(
            run_manifest_path,
            {
                "signature": run_signature,
                "specification": specification,
                "base_revision": revision,
            },
        )

    class CandidateDataset(torch.utils.data.Dataset):
        def __init__(self, examples):
            self.encodings = collections.defaultdict(list)
            self.labels = []

            for start in tqdm(
                range(0, len(examples), 512),
                desc="Tokenizing candidates",
            ):
                group = examples[start:start + 512]

                encoded = tokenizer(
                    [item["text"] for item in group],
                    truncation=True,
                    max_length=MAX_LENGTH,
                    padding=False,
                )

                for key, value in encoded.items():
                    self.encodings[key].extend(value)

                self.labels.extend(
                    item["label"]
                    for item in group
                )

        def __len__(self):
            return len(self.labels)

        def __getitem__(self, index):
            item = {
                key: values[index]
                for key, values in self.encodings.items()
            }
            item["labels"] = self.labels[index]
            return item

    def collate(features):
        batch = tokenizer.pad(
            [
                {
                    key: value
                    for key, value in item.items()
                    if key != "labels"
                }
                for item in features
            ],
            padding=True,
            pad_to_multiple_of=8,
            return_tensors="pt",
        )

        batch["labels"] = torch.tensor(
            [item["labels"] for item in features],
            dtype=torch.long,
        )

        return batch

    complete_path = run / "training_complete_v1.json"

    completed = (
        read_json(complete_path)
        if complete_path.exists()
        else None
    )

    if completed:
        if (
            completed["signature"] != run_signature
            or file_hash(
                model_dir / "model.safetensors"
            ) != completed["model_sha256"]
        ):
            raise ValueError(
                "Completed CWQ answer model does not match this run."
            )

        model = AutoModelForSequenceClassification.from_pretrained(
            str(model_dir),
            local_files_only=True,
        )

        print(
            "Reusing completed CWQ answer recognizer; "
            "training will not repeat."
        )

    else:
        train_dataset = CandidateDataset(train_examples)
        dev_dataset = CandidateDataset(dev_examples)

        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME,
            revision=revision,
            num_labels=2,
            id2label={
                0: "not_answer",
                1: "answer",
            },
            label2id={
                "not_answer": 0,
                "answer": 1,
            },
            problem_type="single_label_classification",
            token=False,
            cache_dir=hf_cache,
        )

        optimizer_steps = (
            math.ceil(len(train_dataset) / TRAIN_BATCH)
            * EPOCHS
        )

        argument_values = {
            "output_dir": str(run / "checkpoints"),
            "num_train_epochs": EPOCHS,
            "per_device_train_batch_size": TRAIN_BATCH,
            "per_device_eval_batch_size": EVAL_BATCH,
            "gradient_accumulation_steps": 1,
            "learning_rate": LEARNING_RATE,

            # Uses warmup_steps, avoiding the earlier API error.
            "warmup_steps": math.ceil(0.1 * optimizer_steps),

            "weight_decay": 0.01,
            "lr_scheduler_type": "linear",
            "fp16": True,
            "bf16": False,
            "max_grad_norm": 1.0,
            "eval_steps": EVAL_STEPS,
            "save_strategy": "steps",
            "save_steps": EVAL_STEPS,
            "save_total_limit": 2,
            "load_best_model_at_end": True,
            "metric_for_best_model": "answer_f1",
            "greater_is_better": True,
            "logging_steps": 100,
            "report_to": [],
            "optim": "adamw_torch",
            "seed": SEED,
            "data_seed": SEED,
            "remove_unused_columns": False,
            "label_names": ["labels"],
            "dataloader_num_workers": 0,
            "eval_accumulation_steps": 16,
        }

        supported = inspect.signature(
            TrainingArguments
        ).parameters

        evaluation_key = (
            "eval_strategy"
            if "eval_strategy" in supported
            else "evaluation_strategy"
        )
        argument_values[evaluation_key] = "steps"

        essential = [
            "output_dir",
            "num_train_epochs",
            "per_device_train_batch_size",
            "learning_rate",
            "warmup_steps",
            "fp16",
            evaluation_key,
            "load_best_model_at_end",
            "metric_for_best_model",
            "label_names",
        ]

        if any(
            name not in supported
            for name in essential
        ):
            raise RuntimeError(
                "Installed TrainingArguments lacks a required option."
            )

        arguments = TrainingArguments(
            **{
                name: value
                for name, value in argument_values.items()
                if name in supported
            }
        )

        def compute_metrics(prediction):
            logits = (
                prediction.predictions[0]
                if isinstance(
                    prediction.predictions,
                    tuple,
                )
                else prediction.predictions
            )

            shifted = (
                logits
                - logits.max(axis=1, keepdims=True)
            )

            probabilities = np.exp(shifted)
            probabilities /= probabilities.sum(
                axis=1,
                keepdims=True,
            )

            result = binary_metrics(
                prediction.label_ids,
                probabilities[:, 1],
                0.5,
            )

            return {
                name: result[name]
                for name in [
                    "accuracy",
                    "balanced_accuracy",
                    "macro_f1",
                    "answer_f1",
                ]
            }

        class FinalStepCallback(TrainerCallback):
            def on_step_end(
                self,
                args,
                state,
                control,
                **kwargs,
            ):
                if state.global_step == state.max_steps:
                    control.should_evaluate = True
                    control.should_save = True
                return control

        trainer_values = {
            "model": model,
            "args": arguments,
            "train_dataset": train_dataset,
            "eval_dataset": dev_dataset,
            "data_collator": collate,
            "compute_metrics": compute_metrics,
            "callbacks": [FinalStepCallback()],
        }

        trainer_parameters = inspect.signature(
            Trainer.__init__
        ).parameters

        if "processing_class" in trainer_parameters:
            trainer_values["processing_class"] = tokenizer
        else:
            trainer_values["tokenizer"] = tokenizer

        trainer = Trainer(**trainer_values)
        trainer.model_accepts_loss_kwargs = False

        checkpoint_dir = run / "checkpoints"

        checkpoint_paths = (
            list(checkpoint_dir.glob("checkpoint-*"))
            if checkpoint_dir.exists()
            else []
        )

        required_checkpoint_files = [
            "trainer_state.json",
            "model.safetensors",
            "optimizer.pt",
            "scheduler.pt",
            "rng_state.pth",
        ]

        valid_checkpoints = [
            path
            for path in checkpoint_paths
            if all(
                (path / name).is_file()
                for name in required_checkpoint_files
            )
        ]

        last_checkpoint = (
            max(
                valid_checkpoints,
                key=lambda path: int(
                    path.name.split("-")[-1]
                ),
            )
            if valid_checkpoints
            else None
        )

        print(
            "Training examples:",
            len(train_examples),
        )
        print(
            "Checkpoint development examples:",
            len(dev_examples),
        )
        print(
            "Expected optimizer steps:",
            optimizer_steps,
        )
        print(
            "Resume checkpoint:",
            last_checkpoint,
        )

        started = time.time()

        training_result = trainer.train(
            resume_from_checkpoint=(
                str(last_checkpoint)
                if last_checkpoint
                else None
            )
        )

        model = trainer.model
        model_dir.mkdir(parents=True, exist_ok=True)

        model.save_pretrained(
            str(model_dir),
            safe_serialization=True,
        )
        tokenizer.save_pretrained(str(model_dir))

        completed = {
            "signature": run_signature,
            "model_sha256": file_hash(
                model_dir / "model.safetensors"
            ),
            "base_revision": revision,
            "best_checkpoint": (
                trainer.state.best_model_checkpoint
            ),
            "best_sample_answer_f1": trainer.state.best_metric,
            "training_minutes_this_invocation": (
                time.time() - started
            ) / 60,
            "training_metrics": training_result.metrics,
            "log_history": trainer.state.log_history,
        }

        save_json(complete_path, completed)

        del trainer, train_dataset, dev_dataset

    training_metrics_path = (
        output / "cwq_answer_recognizer_training_metrics_v1.json"
    )
    save_json(training_metrics_path, completed)

    model.to("cuda").eval()

    def predict(texts):
        values = np.empty(
            len(texts),
            dtype=np.float32,
        )

        order = sorted(
            range(len(texts)),
            key=lambda index: len(texts[index]),
        )

        with torch.inference_mode():
            for start in range(
                0,
                len(order),
                EVAL_BATCH,
            ):
                indices = order[start:start + EVAL_BATCH]

                encoded = tokenizer(
                    [texts[index] for index in indices],
                    padding=True,
                    truncation=True,
                    max_length=MAX_LENGTH,
                    pad_to_multiple_of=8,
                    return_tensors="pt",
                )

                encoded = {
                    key: value.to("cuda")
                    for key, value in encoded.items()
                }

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                ):
                    logits = model(
                        **encoded,
                        return_dict=True,
                    ).logits

                values[indices] = torch.softmax(
                    logits.float(),
                    dim=-1,
                )[:, 1].cpu().numpy()

        return values

    prediction_signature = fingerprint({
        "model_sha256": completed["model_sha256"],
        "dev_sha256": metadata["dev"]["sha256"],
        "max_length": MAX_LENGTH,
        "eval_batch": EVAL_BATCH,
        "code_version": CODE_VERSION,
    })

    print(
        "\nScoring all development candidates "
        "for answer-threshold calibration..."
    )

    groups = score_development(
        paths["dev"],
        metadata["dev"],
        output,
        prediction_signature,
        predict,
    )

    best, sweep = question_threshold_sweep(groups)

    probabilities = np.asarray(
        [
            probability
            for group in groups
            for probability in group["probabilities"]
        ],
        dtype=np.float32,
    )

    labels = np.asarray(
        [
            label
            for group in groups
            for label in group["labels"]
        ],
        dtype=int,
    )

    node_metrics = binary_metrics(
        labels,
        probabilities,
        best["threshold"],
    )

    node_metrics["brier_score"] = float(
        np.mean((probabilities - labels)**2)
    )

    metrics_report = {
        "selected_threshold": best["threshold"],
        "node_classification": node_metrics,
        "development_question_answer_sets": best,
        "development_statistics": metadata["dev"]["statistics"],
        "prediction_signature": prediction_signature,
        "evaluation": (
            "Offline recognition over all canonical BFS "
            "candidates; not online search performance."
        ),
    }

    threshold_report = {
        "selection_split": "development",
        "selected_threshold": best["threshold"],
        "decision_rule": (
            "ANSWER if P(answer) >= selected_threshold"
        ),
        "selection_rule": (
            "Maximize question-macro answer-set F1; "
            "then complete recall; then macro precision; "
            "then higher threshold."
        ),
        "selected_threshold_metrics": best,
        "controller_threshold_unchanged": (
            frozen_controller_threshold
        ),
        "max_hops": MAX_HOPS,
        "maximum_sequence_length": MAX_LENGTH,
        "model_signature": run_signature,
        "model_sha256": completed["model_sha256"],
        "test_used": False,
        "threshold_sweep": sweep,
    }

    recognizer_configuration = {
        "base_model": MODEL_NAME,
        "base_revision": revision,
        "num_labels": 2,
        "class_to_id": {
            "not_answer": 0,
            "answer": 1,
        },
        "maximum_sequence_length": MAX_LENGTH,
        "max_hops": MAX_HOPS,
        "answer_text_version": "webqsp_notebook06_v1",
        "path_policy": "deterministic first shortest BFS path",
        "initial_topic_nodes": (
            "No recognizer candidate without an incoming branch; "
            "included in gold denominators."
        ),
        "selected_answer_threshold": best["threshold"],
        "controller_threshold": frozen_controller_threshold,
        "data_signature": generation["signature"],
        "model_signature": run_signature,
    }

    recognizer_config_path = (
        model_dir / "answer_recognizer_config.json"
    )

    save_json(
        recognizer_config_path,
        recognizer_configuration,
    )

    summary = {
        "model": MODEL_NAME,
        "base_revision": revision,
        "saved_model": str(model_dir),
        "data_signature": generation["signature"],
        "model_signature": run_signature,
        "gpu": torch.cuda.get_device_name(0),
        "transformers_version": transformers.__version__,
        "pytorch_version": torch.__version__,
        "training_minutes_this_invocation": (
            completed["training_minutes_this_invocation"]
        ),
        "train_examples": len(train_examples),
        "checkpoint_dev_examples": len(dev_examples),
        "checkpoint_dev_class_counts": dict(
            collections.Counter(
                str(item["label"])
                for item in dev_examples
            )
        ),
        "full_dev_examples": len(labels),
        "train_statistics": metadata["train"]["statistics"],
        "development_statistics": metadata["dev"]["statistics"],
        "selected_threshold": best["threshold"],
        "development_node_answer_f1": node_metrics["answer_f1"],
        "development_node_precision": node_metrics["answer_precision"],
        "development_node_recall": node_metrics["answer_recall"],
        "development_question_macro_f1": best["macro_f1"],
        "development_question_exact_match": best["exact_match"],
        "development_question_complete_recall": best["complete_recall"],
        "controller_threshold_unchanged": frozen_controller_threshold,
        "maximum_sequence_length": MAX_LENGTH,
        "max_hops": MAX_HOPS,
        "test_used": False,
        "next_step": (
            "Run a development-only online pipeline check "
            "before frozen CWQ test evaluation."
        ),
        "limitations": [
            (
                "Canonical shortest paths can differ from "
                "paths encountered during online search."
            ),
            (
                "Offline answer recognition metrics do not "
                "measure frontier ranking or search cost."
            ),
        ],
    }

    summary_path = (
        output / "cwq_answer_recognizer_summary_v1.json"
    )
    metrics_path = (
        output / "cwq_answer_recognizer_dev_metrics_v1.json"
    )
    threshold_path = (
        output / "cwq_answer_recognizer_threshold_v1.json"
    )

    for path, obj in [
        (summary_path, summary),
        (metrics_path, metrics_report),
        (threshold_path, threshold_report),
    ]:
        save_json(path, obj)

    predictions_path = (
        output / "cwq_answer_recognizer_dev_predictions_v1.jsonl"
    )
    temporary = predictions_path.with_name(
        predictions_path.name + ".partial"
    )

    with open(temporary, "w", encoding="utf-8") as file:
        for group in groups:
            file.write(json.dumps(group) + "\n")

    temporary.replace(predictions_path)

    inventory_paths = [
        summary_path,
        metrics_path,
        threshold_path,
        predictions_path,
        training_metrics_path,
        recognizer_config_path,
        model_dir / "config.json",
        model_dir / "model.safetensors",
    ]

    inventory_path = output / "artifact_inventory_v1.json"

    save_json(
        inventory_path,
        {
            "data_signature": generation["signature"],
            "model_signature": run_signature,
            "files": [
                {
                    "name": path.name,
                    "path": str(path),
                    "bytes": path.stat().st_size,
                    "sha256": file_hash(path),
                }
                for path in inventory_paths
            ],
        },
    )

    print("\n" + "=" * 90)
    print("CWQ ANSWER RECOGNIZER TRAINING AND DEV CALIBRATION COMPLETE")
    print("=" * 90)
    print(json.dumps(summary, indent=2))

    print("\nSaved CWQ answer model:")
    print(model_dir)

    print("\nSEND THESE FOUR REPORTS:")

    for path in [
        summary_path,
        metrics_path,
        threshold_path,
        inventory_path,
    ]:
        print(path)


main()

Mounted at /content/drive
NOTEBOOK 14: CWQ ANSWER-NODE RECOGNIZER
GPU: Tesla T4
Transformers: 5.17.0
PyTorch: 2.11.0+cu130
Model: microsoft/MiniLM-L12-H384-uncased
Frozen controller threshold: 0.853
Selected train/dev questions: 4644 520
Verifying source: train_simple.json
Verifying source: dev_simple.json
Verifying source: entities.txt
Verifying source: relations.txt
train: 200/4,644 questions | 0.1 min
train: 400/4,644 questions | 0.1 min
train: 600/4,644 questions | 0.2 min
train: 800/4,644 questions | 0.3 min
train: 1,000/4,644 questions | 0.4 min
train: 1,200/4,644 questions | 0.5 min
train: 1,400/4,644 questions | 0.6 min
train: 1,600/4,644 questions | 0.6 min
train: 1,800/4,644 questions | 0.7 min
train: 2,000/4,644 questions | 0.8 min
train: 2,200/4,644 questions | 0.8 min
train: 2,400/4,644 questions | 0.9 min
train: 2,600/4,644 questions | 1.0 min
train: 2,800/4,644 questions | 1.0 min
train: 3,000/4,644 questions | 1.1 min
train: 3,200/4,644 questions | 1.2 min
train: 3,400/

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Scoring all development candidates for answer-threshold calibration...
Dev scoring: 25/520 questions | 0.3 min
Dev scoring: 50/520 questions | 0.6 min
Dev scoring: 75/520 questions | 0.9 min
Dev scoring: 100/520 questions | 1.1 min
Dev scoring: 125/520 questions | 1.5 min
Dev scoring: 150/520 questions | 1.8 min
Dev scoring: 175/520 questions | 2.2 min
Dev scoring: 200/520 questions | 2.5 min
Dev scoring: 225/520 questions | 2.9 min
Dev scoring: 250/520 questions | 3.3 min
Dev scoring: 275/520 questions | 3.5 min
Dev scoring: 300/520 questions | 3.9 min
Dev scoring: 325/520 questions | 4.1 min
Dev scoring: 350/520 questions | 4.5 min
Dev scoring: 375/520 questions | 4.8 min
Dev scoring: 400/520 questions | 5.2 min
Dev scoring: 425/520 questions | 5.6 min
Dev scoring: 450/520 questions | 5.9 min
Dev scoring: 475/520 questions | 6.2 min
Dev scoring: 500/520 questions | 6.5 min

CWQ ANSWER RECOGNIZER TRAINING AND DEV CALIBRATION COMPLETE
{
  "model": "microsoft/MiniLM-L12-H384-uncased",
